# S6E4 Predicting Irrigation Need

## Score: .96418

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 200)

In [14]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
ORIGINAL_PATH = PROJECT_ROOT / "original-data" / "irrigation_prediction.csv"
ANCHOR_SUB_PATH = PROJECT_ROOT / "score95787.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SEED = 42
N_SPLITS = 3
USE_ORIGINAL_DATA = True
USE_ANCHOR_FALLBACK = True
FALLBACK_CONF_THRESHOLD = 0.60

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

if USE_ORIGINAL_DATA and ORIGINAL_PATH.exists():
    original_df = pd.read_csv(ORIGINAL_PATH)
    original_df = original_df[[c for c in train_df.columns if c != ID_COL]]
    train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
    print("original shape:", original_df.shape)
else:
    print("original data disabled or missing")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()

original shape: (10000, 20)
train shape: (640000, 21)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0.0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1.0,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2.0,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3.0,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4.0,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [15]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.34
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.015625
Soil_Type                  0.000000
Soil_pH                    0.000000
Soil_Moisture              0.000000
Organic_Carbon             0.000000
Electrical_Conductivity    0.000000
Temperature_C              0.000000
Humidity                   0.000000
Rainfall_mm                0.000000
Sunlight_Hours             0.000000
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [16]:
y = train_df[TARGET].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c != ID_COL]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")

Total features: 19
Categorical   : 8 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical     : 11


In [17]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

classes = sorted(y.unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

y_idx = y.map(class_to_idx).astype(int)

class_counts = y.value_counts()
class_weight_label = {
    cls: len(y) / (len(classes) * class_counts[cls])
    for cls in classes
}
class_weight_idx = {
    class_to_idx[cls]: weight
    for cls, weight in class_weight_label.items()
}

oof_cat = np.zeros((len(X), len(classes)), dtype=np.float64)
oof_lgb = np.zeros((len(X), len(classes)), dtype=np.float64)
oof_xgb = np.zeros((len(X), len(classes)), dtype=np.float64)

test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    y_train_idx, y_valid_idx = y_idx.iloc[train_idx], y_idx.iloc[valid_idx]

    for col in cat_cols:
        X_train[col] = X_train[col].astype("category")
        X_valid[col] = X_valid[col].astype("category")

    X_test_fold = X_test.copy()
    for col in cat_cols:
        X_test_fold[col] = X_test_fold[col].astype("category")

    sample_weight = y_train.map(class_weight_label).values

    cat_model = CatBoostClassifier(
        iterations=700,
        learning_rate=0.06,
        depth=7,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=SEED + fold,
        verbose=100,
        thread_count=-1,
        class_weights=class_weight_label,
    )

    lgb_model = LGBMClassifier(
        n_estimators=700,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multiclass",
        random_state=SEED + fold,
        class_weight=class_weight_idx,
        verbosity=-1,
    )

    xgb_model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        enable_categorical=True,
        random_state=SEED + fold,
        n_jobs=-1,
    )

    cat_model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=80,
    )

    lgb_model.fit(
        X_train,
        y_train_idx,
        eval_set=[(X_valid, y_valid_idx)],
        eval_metric="multi_logloss",
        callbacks=[early_stopping(stopping_rounds=80), log_evaluation(period=0)],
    )

    xgb_model.fit(
        X_train,
        y_train_idx,
        eval_set=[(X_valid, y_valid_idx)],
        sample_weight=sample_weight,
        verbose=False,
    )

    cat_valid = cat_model.predict_proba(X_valid)
    lgb_valid = lgb_model.predict_proba(X_valid)
    xgb_valid = xgb_model.predict_proba(X_valid)

    oof_cat[valid_idx] = cat_valid
    oof_lgb[valid_idx] = lgb_valid
    oof_xgb[valid_idx] = xgb_valid

    cat_pred = np.argmax(cat_valid, axis=1)
    lgb_pred = np.argmax(lgb_valid, axis=1)
    xgb_pred = np.argmax(xgb_valid, axis=1)

    cat_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in cat_pred])
    lgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in lgb_pred])
    xgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in xgb_pred])

    print(
        f"Fold {fold}: cat={cat_score:.6f} lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
    )

    blend_equal = (cat_valid + lgb_valid + xgb_valid) / 3.0
    blend_pred = np.argmax(blend_equal, axis=1)
    fold_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in blend_pred])
    fold_scores.append(fold_score)
    print(f"Fold {fold}: equal-blend balanced_accuracy = {fold_score:.6f}")

    test_cat += cat_model.predict_proba(X_test_fold)
    test_lgb += lgb_model.predict_proba(X_test_fold)
    test_xgb += xgb_model.predict_proba(X_test_fold)

test_cat /= N_SPLITS
test_lgb /= N_SPLITS
test_xgb /= N_SPLITS

best_score = -1.0
best_w = (1.0, 0.0, 0.0)

for w_cat in np.arange(0.30, 0.81, 0.05):
    for w_lgb in np.arange(0.05, 0.56, 0.05):
        w_xgb = 1.0 - w_cat - w_lgb
        if w_xgb < 0.05 or w_xgb > 0.60:
            continue
        blend_oof = w_cat * oof_cat + w_lgb * oof_lgb + w_xgb * oof_xgb
        blend_pred_idx = np.argmax(blend_oof, axis=1)
        blend_pred = [idx_to_class[i] for i in blend_pred_idx]
        score = balanced_accuracy_score(y, blend_pred)
        if score > best_score:
            best_score = score
            best_w = (w_cat, w_lgb, w_xgb)

W_CAT, W_LGB, W_XGB = best_w
test_proba = W_CAT * test_cat + W_LGB * test_lgb + W_XGB * test_xgb

oof_best = W_CAT * oof_cat + W_LGB * oof_lgb + W_XGB * oof_xgb
oof_best_pred = np.argmax(oof_best, axis=1)
oof_best_pred = [idx_to_class[i] for i in oof_best_pred]

cv_score = balanced_accuracy_score(y, oof_best_pred)
print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
print("Fold equal-blend scores:", [round(s, 6) for s in fold_scores])
print("Mean equal-blend score:", round(float(np.mean(fold_scores)), 6))
print(f"Best blend weights: cat={W_CAT:.2f}, lgb={W_LGB:.2f}, xgb={W_XGB:.2f}")
print(f"Best OOF score from weight search: {best_score:.6f}")

0:	learn: 0.9575214	test: 0.9597409	best: 0.9597409 (0)	total: 555ms	remaining: 6m 28s
100:	learn: 0.9659371	test: 0.9667372	best: 0.9667372 (100)	total: 56s	remaining: 5m 32s
200:	learn: 0.9689051	test: 0.9676313	best: 0.9676422 (199)	total: 1m 47s	remaining: 4m 25s
300:	learn: 0.9717058	test: 0.9684289	best: 0.9686132 (297)	total: 2m 37s	remaining: 3m 28s
400:	learn: 0.9733302	test: 0.9685083	best: 0.9689350 (365)	total: 3m 28s	remaining: 2m 35s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.9689349999
bestIteration = 365

Shrink model to first 366 iterations.
Training until validation scores don't improve for 80 rounds
Did not meet early stopping. Best iteration is:
[699]	valid_0's multi_logloss: 0.0581495
Fold 1: cat=0.968928 lgb=0.968263 xgb=0.969152
Fold 1: equal-blend balanced_accuracy = 0.969369
0:	learn: 0.9594435	test: 0.9597505	best: 0.9597505 (0)	total: 572ms	remaining: 6m 39s
100:	learn: 0.9661056	test: 0.9658523	best: 0.9658751 (99)	total: 51.6s	remai

In [18]:
test_conf = np.max(test_proba, axis=1)
test_pred_idx = np.argmax(test_proba, axis=1)
test_pred = pd.Series([idx_to_class[i] for i in test_pred_idx], index=test_df.index)

if USE_ANCHOR_FALLBACK and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)
    anchor_df = anchor_df[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)

    if anchor_labels.isna().any():
        raise ValueError("Anchor submission is missing ids from test set")

    use_anchor_mask = test_conf < FALLBACK_CONF_THRESHOLD
    test_pred.loc[use_anchor_mask] = anchor_labels.loc[use_anchor_mask]
    print(
        f"Anchor fallback applied to {int(use_anchor_mask.sum())} rows "
        f"(threshold={FALLBACK_CONF_THRESHOLD})"
    )
else:
    print("Anchor fallback disabled or anchor file missing")

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred.values,
})
submission.to_csv(SUB_PATH, index=False)

print("Saved:", SUB_PATH.resolve())
submission.head()

Anchor fallback applied to 1309 rows (threshold=0.6)
Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
